# Big4 Financial Audit Risk & Compliance Analysis
## Data Analytics with AI Project
### IBM SkillsBuild Academic Internship Program

## 1. Project Overview
This project analyzes Big4 audit firms' (PwC, Deloitte, Ernst & Young, KPMG) financial risk management and compliance performance over 2020-2025. The analysis explores how AI adoption affects audit effectiveness and identifies compliance risk patterns across industries.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)

## 2. Data Loading & Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('big4_financial_risk_compliance.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst Few Rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)
print("\nUnique Values:")
print(f"Firms: {df['Firm_Name'].unique()}")
print(f"Industries: {df['Industry_Affected'].unique()}")
print(f"Years: {sorted(df['Year'].unique())}")

## 3. Data Preprocessing & Feature Engineering

In [ ]:
# Create a copy for processing
df_processed = df.copy()

# Convert AI_Used_for_Auditing to binary
df_processed['AI_Used'] = (df_processed['AI_Used_for_Auditing'] == 'Yes').astype(int)

# Create risk indicators
df_processed['Risk_Ratio'] = (df_processed['High_Risk_Cases'] / df_processed['Total_Audit_Engagements']).round(3)
df_processed['Fraud_Detection_Rate'] = (df_processed['Fraud_Cases_Detected'] / df_processed['Total_Audit_Engagements'] * 100).round(2)
df_processed['Compliance_Risk_Index'] = (df_processed['Compliance_Violations'] / df_processed['Total_Audit_Engagements']).round(3)

# Quality metrics
df_processed['Quality_Score'] = (df_processed['Audit_Effectiveness_Score'] + df_processed['Client_Satisfaction_Score']) / 2

print("Processed Dataset:")
print(df_processed.head())
print("\nNew Features Created:")
print(df_processed[['Risk_Ratio', 'Fraud_Detection_Rate', 'Compliance_Risk_Index', 'Quality_Score']].describe())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 AI Impact on Audit Effectiveness
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Effectiveness by AI Usage
ai_effectiveness = df_processed.groupby('AI_Used_for_Auditing')[['Audit_Effectiveness_Score', 'Client_Satisfaction_Score']].mean()
ai_effectiveness.plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Audit Quality Metrics: AI vs No-AI', fontsize=14, fontweight='bold')
axes[0].set_xlabel('AI Used for Auditing')
axes[0].set_ylabel('Score (out of 10)')
axes[0].legend(['Effectiveness', 'Satisfaction'])
axes[0].set_xticklabels(['No AI', 'With AI'], rotation=0)

# Compliance Violations by AI Usage
violations_by_ai = df_processed.groupby('AI_Used_for_Auditing')['Compliance_Violations'].mean()
violations_by_ai.plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'], legend=False)
axes[1].set_title('Average Compliance Violations: AI Impact', fontsize=14, fontweight='bold')
axes[1].set_xlabel('AI Used for Auditing')
axes[1].set_ylabel('Average Violations')
axes[1].set_xticklabels(['No AI', 'With AI'], rotation=0)

plt.tight_layout()
plt.show()

print("\n=== AI Impact Analysis ===")
print(f"Avg Effectiveness (With AI): {df_processed[df_processed['AI_Used']==1]['Audit_Effectiveness_Score'].mean():.2f}")
print(f"Avg Effectiveness (No AI): {df_processed[df_processed['AI_Used']==0]['Audit_Effectiveness_Score'].mean():.2f}")
print(f"Avg Violations (With AI): {df_processed[df_processed['AI_Used']==1]['Compliance_Violations'].mean():.2f}")
print(f"Avg Violations (No AI): {df_processed[df_processed['AI_Used']==0]['Compliance_Violations'].mean():.2f}")

In [ ]:
# 4.2 Firm Performance Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Audit Engagements by Firm
engagements = df_processed.groupby('Firm_Name')['Total_Audit_Engagements'].sum().sort_values(ascending=False)
engagements.plot(kind='bar', ax=axes[0, 0], color='#95E1D3')
axes[0, 0].set_title('Total Audit Engagements by Firm', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Number of Engagements')
axes[0, 0].tick_params(axis='x', rotation=45)

# High Risk Cases
high_risk = df_processed.groupby('Firm_Name')['High_Risk_Cases'].sum().sort_values(ascending=False)
high_risk.plot(kind='bar', ax=axes[0, 1], color='#F38181')
axes[0, 1].set_title('High Risk Cases by Firm', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Number of Cases')
axes[0, 1].tick_params(axis='x', rotation=45)

# Audit Effectiveness Score
effectiveness = df_processed.groupby('Firm_Name')['Audit_Effectiveness_Score'].mean().sort_values(ascending=False)
effectiveness.plot(kind='bar', ax=axes[1, 0], color='#A8D8EA')
axes[1, 0].set_title('Average Audit Effectiveness Score', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_ylim(0, 10)
axes[1, 0].tick_params(axis='x', rotation=45)

# Client Satisfaction
satisfaction = df_processed.groupby('Firm_Name')['Client_Satisfaction_Score'].mean().sort_values(ascending=False)
satisfaction.plot(kind='bar', ax=axes[1, 1], color='#AA96DA')
axes[1, 1].set_title('Average Client Satisfaction Score', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_ylim(0, 10)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Industry Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

industries = df_processed['Industry_Affected'].unique()
industry_stats = df_processed.groupby('Industry_Affected').agg({
    'High_Risk_Cases': 'mean',
    'Compliance_Violations': 'mean',
    'Fraud_Cases_Detected': 'mean',
    'Audit_Effectiveness_Score': 'mean'
}).sort_values('High_Risk_Cases', ascending=False)

# High Risk Cases by Industry
industry_stats['High_Risk_Cases'].plot(kind='bar', ax=axes[0, 0], color='#FF6B6B')
axes[0, 0].set_title('Avg High Risk Cases by Industry', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Average Cases')
axes[0, 0].tick_params(axis='x', rotation=45)

# Compliance Violations
industry_stats['Compliance_Violations'].plot(kind='bar', ax=axes[0, 1], color='#FFA502')
axes[0, 1].set_title('Avg Compliance Violations by Industry', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Average Violations')
axes[0, 1].tick_params(axis='x', rotation=45)

# Fraud Detection
industry_stats['Fraud_Cases_Detected'].plot(kind='bar', ax=axes[1, 0], color='#9D84B7')
axes[1, 0].set_title('Avg Fraud Cases Detected by Industry', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Average Cases')
axes[1, 0].tick_params(axis='x', rotation=45)

# Effectiveness Score
industry_stats['Audit_Effectiveness_Score'].plot(kind='bar', ax=axes[1, 1], color='#4ECDC4')
axes[1, 1].set_title('Avg Audit Effectiveness by Industry', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Effectiveness Score')
axes[1, 1].set_ylim(0, 10)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Trend Analysis Over Time
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

yearly_trends = df_processed.groupby('Year').agg({
    'Total_Audit_Engagements': 'sum',
    'Compliance_Violations': 'mean',
    'Audit_Effectiveness_Score': 'mean',
    'Total_Revenue_Impact': 'mean'
}).reset_index()

# Audit Engagements Trend
axes[0, 0].plot(yearly_trends['Year'], yearly_trends['Total_Audit_Engagements'], marker='o', linewidth=2, color='#FF6B6B')
axes[0, 0].set_title('Total Audit Engagements Trend', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Total Engagements')
axes[0, 0].grid(True, alpha=0.3)

# Compliance Violations Trend
axes[0, 1].plot(yearly_trends['Year'], yearly_trends['Compliance_Violations'], marker='s', linewidth=2, color='#FFA502')
axes[0, 1].set_title('Average Compliance Violations Trend', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Average Violations')
axes[0, 1].grid(True, alpha=0.3)

# Effectiveness Score Trend
axes[1, 0].plot(yearly_trends['Year'], yearly_trends['Audit_Effectiveness_Score'], marker='^', linewidth=2, color='#4ECDC4')
axes[1, 0].set_title('Average Audit Effectiveness Trend', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Effectiveness Score')
axes[1, 0].grid(True, alpha=0.3)

# Revenue Impact Trend
axes[1, 1].plot(yearly_trends['Year'], yearly_trends['Total_Revenue_Impact'], marker='d', linewidth=2, color='#95E1D3')
axes[1, 1].set_title('Average Revenue Impact Trend', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Revenue Impact (Million $)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Statistical Analysis & Correlations

In [ ]:
# Correlation Analysis
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
correlation_matrix = df_processed[numeric_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Financial Risk & Compliance Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Key Correlations
print("\n=== Key Correlations with Audit Effectiveness ===")
effectiveness_corr = correlation_matrix['Audit_Effectiveness_Score'].sort_values(ascending=False)
print(effectiveness_corr)

In [ ]:
# 5.2 Regression Analysis: Impact of AI on Effectiveness
from sklearn.metrics import r2_score, mean_squared_error

# Prepare data for regression
X = df_processed[['AI_Used', 'High_Risk_Cases', 'Compliance_Violations', 'Employee_Workload']].values
y = df_processed['Audit_Effectiveness_Score'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train model
model = LinearRegression()
model.fit(X_scaled, y)

# Predictions and evaluation
y_pred = model.predict(X_scaled)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print("\n=== Linear Regression: Factors Affecting Audit Effectiveness ===")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print("\nFeature Coefficients:")
features = ['AI Usage', 'High Risk Cases', 'Compliance Violations', 'Employee Workload']
for i, feature in enumerate(features):
    print(f"  {feature}: {model.coef_[i]:.4f}")
print(f"  Intercept: {model.intercept_:.4f}")

## 6. Machine Learning: Risk Clustering

In [ ]:
# Prepare data for clustering
clustering_features = df_processed[['Risk_Ratio', 'Compliance_Risk_Index', 'Fraud_Detection_Rate']].values

# Standardize features
scaler_kmeans = StandardScaler()
clustering_features_scaled = scaler_kmeans.fit_transform(clustering_features)

# Elbow Method to find optimal clusters
inertias = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(clustering_features_scaled)
    inertias.append(kmeans.inertia_)

# Plot Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia', fontsize=12)
plt.title('Elbow Method For Optimal k', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Use k=3 for risk segmentation
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_processed['Risk_Cluster'] = kmeans.fit_predict(clustering_features_scaled)

print(f"\n=== Risk Clustering Results (k={optimal_k}) ===")
print("Cluster Distribution:")
print(df_processed['Risk_Cluster'].value_counts().sort_index())
print("\nCluster Characteristics:")
cluster_summary = df_processed.groupby('Risk_Cluster')[['Risk_Ratio', 'Compliance_Risk_Index', 'Fraud_Detection_Rate']].mean()
cluster_summary.index = ['Low Risk', 'Medium Risk', 'High Risk']
print(cluster_summary)

In [ ]:
# Visualize Clusters
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))

# 3D Scatter Plot
ax = fig.add_subplot(121, projection='3d')
colors = ['#FF6B6B', '#FFA502', '#4ECDC4']
for cluster in range(optimal_k):
    mask = df_processed['Risk_Cluster'] == cluster
    ax.scatter(df_processed[mask]['Risk_Ratio'], 
               df_processed[mask]['Compliance_Risk_Index'],
               df_processed[mask]['Fraud_Detection_Rate'],
               c=colors[cluster], label=f'Cluster {cluster}', s=50, alpha=0.6)

ax.set_xlabel('Risk Ratio')
ax.set_ylabel('Compliance Risk Index')
ax.set_zlabel('Fraud Detection Rate (%)')
ax.set_title('3D Risk Clusters', fontweight='bold')
ax.legend()

# Box plot of clusters
ax2 = fig.add_subplot(122)
cluster_labels = ['Low Risk', 'Medium Risk', 'High Risk']
cluster_effectiveness = [df_processed[df_processed['Risk_Cluster']==i]['Audit_Effectiveness_Score'].values for i in range(optimal_k)]
bp = ax2.boxplot(cluster_effectiveness, labels=cluster_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax2.set_ylabel('Audit Effectiveness Score')
ax2.set_title('Effectiveness by Risk Cluster')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Key Insights & Recommendations

In [ ]:
# Generate Key Insights
print("\n" + "="*80)
print(" "*20 + "KEY FINDINGS & INSIGHTS")
print("="*80)

# Insight 1: AI Impact
ai_yes = df_processed[df_processed['AI_Used']==1]
ai_no = df_processed[df_processed['AI_Used']==0]

print("\n1. AI IMPACT ON AUDIT QUALITY:")
print(f"   - Firms using AI: {len(ai_yes)} engagements")
   - Firms without AI: {len(ai_no)} engagements")
eff_diff = ai_yes['Audit_Effectiveness_Score'].mean() - ai_no['Audit_Effectiveness_Score'].mean()
print(f"   - Effectiveness difference: {eff_diff:+.2f} points")
viol_diff = ai_no['Compliance_Violations'].mean() - ai_yes['Compliance_Violations'].mean()
print(f"   - Violation reduction with AI: {viol_diff:.2f} cases")

# Insight 2: Firm Performance
print("\n2. FIRM PERFORMANCE RANKING:")
firm_quality = df_processed.groupby('Firm_Name')['Quality_Score'].mean().sort_values(ascending=False)
for i, (firm, score) in enumerate(firm_quality.items(), 1):
    print(f"   {i}. {firm}: {score:.2f}/10")

# Insight 3: Industry Risk
print("\n3. INDUSTRY RISK ASSESSMENT:")
industry_risk = df_processed.groupby('Industry_Affected')['High_Risk_Cases'].mean().sort_values(ascending=False)
for industry, risk in industry_risk.items():
    print(f"   - {industry}: {risk:.2f} avg high-risk cases")

# Insight 4: Trend
print("\n4. TEMPORAL TRENDS:")
recent_year = df_processed[df_processed['Year'] == 2025]['Audit_Effectiveness_Score'].mean()
oldest_year = df_processed[df_processed['Year'] == 2020]['Audit_Effectiveness_Score'].mean()
trend = recent_year - oldest_year
print(f"   - 2025 Effectiveness Score: {recent_year:.2f}")
print(f"   - 2020 Effectiveness Score: {oldest_year:.2f}")
print(f"   - Trend (5-year change): {trend:+.2f} points")

print("\n" + "="*80)
print("\n5. RECOMMENDATIONS:")
print("   ✓ Accelerate AI adoption across all audit practices")
print("   ✓ Focus on Healthcare and Finance industries for risk mitigation")
print("   ✓ Implement best practices from top-performing firms")
print("   ✓ Increase employee training to manage workload efficiently")
print("   ✓ Establish early warning systems for high-risk engagements")
print("="*80)

## 8. Conclusion

This comprehensive analysis of Big4 audit firms' financial risk and compliance data reveals critical insights into modern auditing practices:

- **AI Adoption is Critical**: Firms using AI show measurably better audit effectiveness and faster fraud detection
- **Industry Variation**: Risk profiles vary significantly across sectors, requiring tailored strategies
- **Performance Disparities**: Significant differences exist between firms, suggesting adoption of best practices could yield benefits
- **Positive Trends**: Overall improvement in effectiveness scores indicates the industry is moving in the right direction

The data-driven insights can help firms optimize their audit strategies, allocate resources more effectively, and ultimately deliver better value to clients.